In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
def create_spatial_model(file_path='CGWB_data_main_cleaned.csv'):
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: {file_path} not found.")
        return None, None

    id_vars = ['STATE', 'DISTRICT', 'LAT', 'LON', 'SITE_TYPE', 'WLCODE']
    df_long = pd.melt(df, id_vars=id_vars, var_name='Date', value_name='Water_Level')

    df_long['Date'] = pd.to_datetime(df_long['Date'], errors='coerce')
    df_long.dropna(subset=['Water_Level', 'LAT', 'LON', 'Date'], inplace=True)

    df_recent = df_long.loc[df_long.groupby('WLCODE')['Date'].idxmax()]

    X = df_recent[['LAT', 'LON']]
    y = df_recent['Water_Level']

    knn_model = KNeighborsRegressor(n_neighbors=5, weights='distance')
    knn_model.fit(X, y)

    print("✅ Spatial model trained (KNN)")
    return knn_model, df_long


In [3]:
def prepare_lstm_data(yearly_avg, seq_len=3):
    X, y = [], []
    vals = yearly_avg['Water_Level'].values

    for i in range(len(vals) - seq_len):
        X.append(vals[i:i + seq_len])
        y.append(vals[i + seq_len])

    return np.array(X), np.array(y)

In [4]:
def build_cnn_lstm(seq_len):
    model = Sequential([
        Conv1D(32, kernel_size=2, activation='relu', input_shape=(seq_len, 1)),
        MaxPooling1D(pool_size=1),
        LSTM(64, return_sequences=False),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model

In [5]:
def cnn_lstm_predict(yearly_avg, future_year, seq_len=3, output_dir="outputs"):

    X, y = prepare_lstm_data(yearly_avg, seq_len)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X).reshape(X.shape[0], seq_len, 1)

    model = build_cnn_lstm(seq_len)

    # Train model
    history = model.fit(X_scaled, y, epochs=100, batch_size=4, verbose=0)

    # ----- ACCURACY METRICS -----
    y_pred_train = model.predict(X_scaled, verbose=0)
    mse = mean_squared_error(y, y_pred_train)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, y_pred_train)
    r2 = r2_score(y, y_pred_train)

    print("\n📊 TRAINING ACCURACY")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"R²   : {r2:.4f}")

    # ----- TRAINING LOSS PLOT -----
    os.makedirs(output_dir, exist_ok=True)

    plt.figure(figsize=(6,4))
    plt.plot(history.history['loss'])
    plt.title("CNN + LSTM Model Loss Curve")
    plt.xlabel("Epochs")
    plt.ylabel("Loss (MSE)")
    loss_plot_path = os.path.join(output_dir, "training_loss.png")
    plt.savefig(loss_plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    # ----- FUTURE PREDICTIONS -----
    future_predictions = {}
    last_seq = X[-1]
    last_year = yearly_avg["Year"].max()

    for year in range(last_year + 1, future_year + 1):
        seq_scaled = scaler.transform(last_seq.reshape(1, -1)).reshape(1, seq_len, 1)
        next_pred = model.predict(seq_scaled, verbose=0)[0][0]

        future_predictions[year] = next_pred
        last_seq = np.append(last_seq[1:], next_pred)

    return future_predictions, loss_plot_path, (mse, rmse, mae, r2)

In [6]:
def predict_until_year(latitude, longitude, future_year, model, df_long, output_dir="outputs"):
    if model is None or df_long is None:
        return "Model unavailable."

    df = df_long

    input_data = pd.DataFrame([[latitude, longitude]], columns=['LAT', 'LON'])
    distances, indices = model.kneighbors(input_data, n_neighbors=1)

    nearest_lat, nearest_lon = model._fit_X[indices[0][0]]
    nearest_well = df[(df['LAT'] == nearest_lat) & (df['LON'] == nearest_lon)]['WLCODE'].iloc[0]

    ts = df[df['WLCODE'] == nearest_well].copy()
    ts["Year"] = ts["Date"].dt.year
    yearly_avg = ts.groupby("Year")["Water_Level"].mean().reset_index()

    # Predictions + accuracy + training loss image
    predictions, loss_img, scores = cnn_lstm_predict(yearly_avg, future_year)

    # plotting
    plt.figure(figsize=(8, 5))
    plt.plot(yearly_avg["Year"], yearly_avg["Water_Level"], marker="o", label="Historical")
    plt.plot(list(predictions.keys()), list(predictions.values()), marker="o",
             linestyle="--", color="red", label="Predicted")
    plt.xlabel("Year")
    plt.ylabel("Groundwater Level (m)")
    plt.title(f"Groundwater Prediction (CNN+LSTM) for Well {nearest_well}")
    plt.legend()
    plt.grid(True)

    os.makedirs(output_dir, exist_ok=True)

    plot_path = os.path.join(output_dir, f"prediction_{nearest_well}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    return nearest_well, predictions, plot_path, loss_img, scores


In [7]:
spatial_model, df_long = create_spatial_model('CGWB_data_main_cleaned.csv')

if spatial_model:
    lat = float(input("Enter latitude: "))
    lon = float(input("Enter longitude: "))
    year = int(input("Predict groundwater level up to year: "))

    well, preds, plot_file, loss_file, acc = predict_until_year(lat, lon, year, spatial_model, df_long)

    print("\n--- 📌 Groundwater Future Prediction ---")
    print(f"Nearest Well: {well}")

    for yr, lvl in preds.items():
        print(f"{yr} → {lvl:.2f} meters")

    print("\n--- 📊 Accuracy Scores ---")
    print(f"MSE  : {acc[0]:.3f}")
    print(f"RMSE : {acc[1]:.3f}")
    print(f"MAE  : {acc[2]:.3f}")
    print(f"R²   : {acc[3]:.3f}")

    print("\n📁 Prediction Image Saved:", plot_file)
    print("📁 Training Loss Image Saved:", loss_file)

✅ Spatial model trained (KNN)


c:\Users\mudit\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



📊 TRAINING ACCURACY
MSE  : 0.2211
RMSE : 0.4702
MAE  : 0.2588
R²   : 0.5200

--- 📌 Groundwater Future Prediction ---
Nearest Well: W30848
2018 → 2.40 meters
2019 → 2.08 meters
2020 → 1.87 meters
2021 → 1.77 meters
2022 → 1.76 meters
2023 → 1.80 meters
2024 → 1.85 meters
2025 → 1.88 meters
2026 → 1.87 meters
2027 → 1.84 meters
2028 → 1.83 meters
2029 → 1.82 meters
2030 → 1.83 meters
2031 → 1.84 meters
2032 → 1.84 meters
2033 → 1.84 meters
2034 → 1.84 meters
2035 → 1.84 meters

--- 📊 Accuracy Scores ---
MSE  : 0.221
RMSE : 0.470
MAE  : 0.259
R²   : 0.520

📁 Prediction Image Saved: outputs\prediction_W30848.png
📁 Training Loss Image Saved: outputs\training_loss.png


In [ ]:
# actual vs predicted plot groundwater levels
# scater plot of actual vs predicted groundwater levels
# residual box plot of actual vs predicted groundwater levels
# future groundwater level prediction trend plot